# Policy gradient

- Finite Diﬀerence Policy Gradient
- Monte-Carlo Policy Gradient
- Actor-Critic Policy Gradient

## Policy-Based Reinforcement Learning

### From Value Functions to Policy

Previously, we approximated value functions with parameters $\theta$:

$$V_\theta(s) \approx V^\pi(s)$$
$$Q_\theta(s, a) \approx Q^\pi(s, a)$$

The policy was then **derived indirectly** from these value functions — e.g. via $\varepsilon$-greedy: take the action with highest $Q$, with occasional random exploration.

### Direct Policy Parametrisation

Now we skip the value function and **parametrise the policy directly**:

$$\pi_\theta(s, a) = \mathbb{P}[a \mid s, \theta]$$

This reads: *"given state $s$ and parameters $\theta$, what is the probability of taking action $a$?"*

Instead of learning *"how good is this state/action?"*, we learn *"what action should I take in this state?"* directly.

> The policy outputs a **probability distribution** over actions — stochastic by nature, unlike $\varepsilon$-greedy which is deterministic with added noise.

## Scope

We stay in the **model-free** setting — no knowledge of environment dynamics ($P$, $R$). The agent learns purely from sampled experience.

## Value-Based and Policy-Based RL

Three families of RL algorithms, differing in what they learn:

| | Value Function | Policy |
|---|---|---|
| **Value-Based** | ✅ Learnt | Implicit (e.g. $\varepsilon$-greedy) |
| **Policy-Based** | ❌ None | ✅ Learnt |
| **Actor-Critic** | ✅ Learnt | ✅ Learnt |

### Value-Based
Learns $Q(s,a)$ or $V(s)$, then derives a policy implicitly — no explicit policy is stored.

### Policy-Based
Directly learns $\pi_\theta(s,a)$. No value function at all. Examples: REINFORCE.

### Actor-Critic
Learns both. The **critic** estimates the value function (to reduce variance), the **actor** updates the policy directly. Best of both worlds.

## Advantages of Policy-Based RL

### Advantages

- **Better convergence properties** — optimising the policy directly tends to be more stable than value-based methods, which can oscillate or diverge.
- **Effective in high-dimensional or continuous action spaces** — value-based methods need $\arg\max_a Q(s,a)$, which is intractable when actions are continuous. Policy-based methods output actions directly.
- **Can learn stochastic policies** — sometimes the optimal policy is genuinely random (e.g. in partially observable or adversarial settings). $\varepsilon$-greedy is a hack; $\pi_\theta$ can learn the right distribution naturally.

### Disadvantages

- **Converges to a local optimum** — gradient ascent on the policy objective is not convex; no guarantee of finding the global best.
- **High variance, sample-inefficient** — policy evaluation requires rolling out full episodes, and the gradient estimates are noisy. This is the main motivation for Actor-Critic methods.

## Example: Rock-Paper-Scissors

Classic two-player zero-sum game: scissors beats paper, rock beats scissors, paper beats rock.

### Why deterministic policies fail

In the *iterated* version (many rounds against the same opponent), any deterministic policy is exploitable:
if you always play rock, your opponent learns this and always plays paper.

### The optimal policy is stochastic

Playing each action with probability $\frac{1}{3}$ is the **Nash equilibrium** — no opponent can gain an edge
regardless of their strategy, because your moves are unpredictable.

> This is a key motivation for policy-based RL: value-based methods (e.g. $\varepsilon$-greedy) cannot
> represent a truly uniform random policy. Direct policy parametrisation can.

## Example: Aliased Gridworld (1)

A gridworld where the agent cannot distinguish the two grey states (partial observability).
The money ($) is the goal; skulls are traps.

<img src="../image-106.png" width="400px"/>



### Feature representation

We work with features $\phi(s, a)$ instead of raw states.

Example feature:

$$\phi(s, a) = \mathbf{1}(\text{wall to N}, a = \text{move E})$$

The grey states are the two cells that look identical to the agent — same features $\phi(s,a)$  (e.g. wall to the north in both cases).

The problem: one grey state requires going left to reach the money, the other requires going right. But since the agent can't tell them apart, it must apply the same policy to both.

### Two approaches

**Value-based** — learn an approximate action-value function:

$$Q_\theta(s, a) = f(\phi(s, a), \theta)$$

The policy is then implicit: take $\arg\max_a Q_\theta(s, a)$.

**Policy-based** — directly learn a policy:

$$\pi_\theta(s, a) = g(\phi(s, a), \theta)$$

Outputs a probability distribution over actions given the features.

> Because the two grey states look identical, a deterministic policy must do the
> same thing in both — which may be suboptimal. A stochastic policy can hedge,
> which is the advantage shown in the next slide.

### Example: Aliased Gridworld (2) — Why Deterministic Policies Fail

<img src="../image-107.png" width="400px" />

#### The aliasing trap

A deterministic policy must do the **same thing in both grey states** (they look identical).
So it must pick one of:

- always move **W** in both grey states, or
- always move **E** in both grey states

Either way, one grey state will send the agent in the wrong direction — away from the money
and potentially into a skull. It can get **stuck in a loop** and never reach the goal.

#### Value-based RL makes this worse

Greedy or $\varepsilon$-greedy policies are **near-deterministic** — they almost always pick
$\arg\max_a Q(s,a)$. Since both grey states map to the same $Q$ values (identical features),
the agent commits to one direction and traverses the corridor back and forth for a long time
without ever reaching the money.

#### The fix

A **stochastic policy** (policy-based RL) can assign probability 0.5 to W and 0.5 to E in the
grey states — guaranteed to eventually reach the money from either grey state, just by chance.


### Example: Aliased Gridworld (3) — Optimal Stochastic Policy

<img src="../image-108.png" width="400px" />



#### Solution: randomise in aliased states

Since both grey states have walls to the N and S, the optimal stochastic policy simply
plays 50/50:

$$\pi_\theta(\text{wall to N and S, move E}) = 0.5$$
$$\pi_\theta(\text{wall to N and S, move W}) = 0.5$$

From either grey state, the agent will eventually pick the correct direction by chance
and reach the money in a few steps — with high probability.

#### Key takeaway

Policy-based RL **can learn this stochastic policy directly** via gradient ascent on $\theta$.
Value-based RL cannot — greedy action selection always collapses to a deterministic choice,
which is provably suboptimal here.

> Partial observability (aliasing) is a